### Setup

In [ ]:
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device]: {DEVICE}")

In [ ]:
config = {
    "correlation_mode": "max",
    "correlation_loss": "full",
    "learning_rate": 1e-3,
    "architecture_type": "cnn",
    "epochs": 10,
    "batch_size": 64,
}

run = wandb.init(
    project="cnn-redundancy-reduction",
    config=config,
    name=(
        f"{config['architecture_type']}-{config['correlation_mode']}"
        f"-{config['correlation_loss']}"
        f"-lr_{config['learning_rate']}"
    ),
)

CORRELATION_MODE  = run.config["correlation_mode"]
CORRELATION_LOSS  = run.config["correlation_loss"]
LEARNING_RATE     = run.config["learning_rate"]
ARCHITECTURE_TYPE = run.config["architecture_type"]
EPOCHS            = run.config["epochs"]
BATCH_SIZE        = run.config["batch_size"]

### Dataset

In [ ]:
DATA_DIR    = "./data"
NUM_WORKERS = 2

transform = transforms.Compose([
    transforms.ToTensor(),
])

train_ds = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=transform)
test_ds = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("[train]:", len(train_ds), "[test]:", len(test_ds))
xb, yb = next(iter(train_loader))
print("[batch]:", tuple(xb.shape), tuple(yb.shape), xb.dtype)

### Architectures

In [ ]:
class BasicCNN(nn.Module):
    def __init__(self, in_ch=3, num_classes=10, width=64):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, width, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(width)
        self.conv2 = nn.Conv2d(width, width, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(width)
        self.pool1 = nn.MaxPool2d(2)

        self.conv3 = nn.Conv2d(width, width * 2, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn3   = nn.BatchNorm2d(width * 2)
        self.conv4 = nn.Conv2d(width * 2, width * 2, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn4   = nn.BatchNorm2d(width * 2)
        self.pool2 = nn.MaxPool2d(2)

        self.conv5 = nn.Conv2d(width * 2, width * 4, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn5   = nn.BatchNorm2d(width * 4)
        self.pool3 = nn.MaxPool2d(2)

        self.act     = nn.ReLU(inplace=True)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(width * 4, num_classes)

    def forward(self, x):
        convs = []

        c1, c1k = self.conv1(x), self.conv1.kernel_size
        convs.append(("cnn.conv1", c1, c1k))
        x = self.act(self.bn1(c1))

        c2, c2k = self.conv2(x), self.conv2.kernel_size
        convs.append(("cnn.conv2", c2, c2k))
        x = self.act(self.bn2(c2))
        x = self.pool1(x)

        c3, c3k = self.conv3(x), self.conv3.kernel_size
        convs.append(("cnn.conv3", c3, c3k))
        x = self.act(self.bn3(c3))

        c4, c4k = self.conv4(x), self.conv4.kernel_size
        convs.append(("cnn.conv4", c4, c4k))
        x = self.act(self.bn4(c4))
        x = self.pool2(x)

        c5, c5k = self.conv5(x), self.conv5.kernel_size
        convs.append(("cnn.conv5", c5, c5k))
        x = self.act(self.bn5(c5))
        x = self.pool3(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        logits = self.fc(x)
        return logits, convs

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)

        self.skip_conv = None
        self.skip_bn   = None
        if stride != 1 or in_ch != out_ch:
            self.skip_conv = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, padding=0, bias=False)
            self.skip_bn   = nn.BatchNorm2d(out_ch)

        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        convs = []

        identity = x
        if self.skip_conv is not None and self.skip_bn is not None:
            ds, dsk = self.skip_conv(x), self.skip_conv.kernel_size
            convs.append(("downsample", ds, dsk))
            identity = self.skip_bn(ds)

        c1, c1k = self.conv1(x), self.conv1.kernel_size
        convs.append(("conv1", c1, c1k))
        out     = self.act(self.bn1(c1))

        c2, c2k = self.conv2(out), self.conv2.kernel_size
        convs.append(("conv2", c2, c2k))
        out     = self.bn2(c2)

        out = out + identity
        out = self.act(out)

        return out, convs


class ResNet(nn.Module):
    def __init__(self, in_ch=3, num_classes=10, widths=(64, 128, 256, 512)):
        super().__init__()
        w1, w2, w3, w4 = widths

        self.stem_conv = nn.Conv2d(in_ch, w1, kernel_size=3, stride=1, padding=1, bias=False)
        self.stem_bn   = nn.BatchNorm2d(w1)
        self.stem_act  = nn.ReLU(inplace=True)

        self.layer1 = nn.ModuleList([ResBlock(w1, w1, 1), ResBlock(w1, w1, 1)])
        self.layer2 = nn.ModuleList([ResBlock(w1, w2, 2), ResBlock(w2, w2, 1)])
        self.layer3 = nn.ModuleList([ResBlock(w2, w3, 2), ResBlock(w3, w3, 1)])
        self.layer4 = nn.ModuleList([ResBlock(w3, w4, 2), ResBlock(w4, w4, 1)])

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(w4, num_classes)

    def forward(self, x):
        stem, stemk = self.stem_conv(x), self.stem_conv.kernel_size
        x = self.stem_act(self.stem_bn(stem))

        convs = []
        convs.append(("resnet.stem", stem, stemk))

        for i, block in enumerate(self.layer1):
            x, bconvs = block(x)
            for name, c, k in bconvs:
                convs.append((f"resnet.layer1.block{i}.{name}", c, k))
        for i, block in enumerate(self.layer2):
            x, bconvs = block(x)
            for name, c, k in bconvs:
                convs.append((f"resnet.layer2.block{i}.{name}", c, k))
        for i, block in enumerate(self.layer3):
            x, bconvs = block(x)
            for name, c, k in bconvs:
                convs.append((f"resnet.layer3.block{i}.{name}", c, k))
        for i, block in enumerate(self.layer4):
            x, bconvs = block(x)
            for name, c, k in bconvs:
                convs.append((f"resnet.layer4.block{i}.{name}", c, k))

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits = self.fc(x)
        return logits, convs


In [ ]:
class VGG16(nn.Module):
    def __init__(self, in_ch=3, num_classes=10):
        super().__init__()
        cfg = [
            64,
            64,
            "M",
            128,
            128,
            "M",
            256,
            256,
            256,
            "M",
            512,
            512,
            512,
            "M",
            512,
            512,
            512,
            "M",
        ]

        self.layers = nn.ModuleList()
        ch = in_ch
        for v in cfg:
            if v == "M":
                self.layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                out_ch = int(v)
                self.layers.append(nn.Conv2d(ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False))
                self.layers.append(nn.BatchNorm2d(out_ch))
                self.layers.append(nn.ReLU(inplace=True))
                ch = out_ch

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        convs = []

        for i, layer in enumerate(self.layers):
            if isinstance(layer, nn.Conv2d):
                c, ck = layer(x), layer.kernel_size
                x = c
                convs.append((f"vgg16.layers.{i}", c, ck))
            else:
                x = layer(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits = self.fc(x)
        return logits, convs


In [ ]:
def get_autoencoder_type(autoencoder_type: str):
    match autoencoder_type:
        case "cnn":
            return BasicCNN
        case "resnet":
            return ResNet
        case "vgg16":
            return VGG16
        case _:
            raise ValueError(
                f"Unknown autoencoder_type={autoencoder_type!r}."
            )

### Criterion

In [ ]:
class RedundancyLoss(nn.Module):
    def __init__(
        self,
        lambda_strength: float,
        correlation_mode: str = "max",
        correlation_loss: str = "full",
        eps: float = 1e-6,
    ):
        super().__init__()
        self.lambda_strength = float(lambda_strength)
        self.correlation_mode = correlation_mode
        self.correlation_loss = correlation_loss
        self.eps = float(eps)

        self.ce = nn.CrossEntropyLoss()

        self.last_ce_loss = torch.tensor(0.0)
        self.last_corr_total = torch.tensor(0.0)

    def forward(
        self,
        y: torch.Tensor,
        y_pred: torch.Tensor,
        fm_list: list,
    ) -> torch.Tensor:
        ce_loss = self.ce(y_pred, y)

        corr_total = y_pred.new_zeros(())
        if fm_list:
            for fm_item in fm_list:
                feature_map, kernel_radius = self._parse_feature_map(fm_item)

                corr_mat = self.luca_fn(
                    feature_map=feature_map,
                    kernel_radius=kernel_radius,
                    mode=self.correlation_mode,
                    eps=self.eps,
                )

                corr_total = corr_total + self._reduce_corr_matrix(
                    corr_mat,
                    correlation_loss=self.correlation_loss,
                )

        self.last_ce_loss = ce_loss.detach()
        self.last_corr_total = corr_total.detach()

        return ce_loss + (self.lambda_strength * corr_total)

    @staticmethod
    def _parse_feature_map(fm_item) -> tuple[torch.Tensor, int]:
        if isinstance(fm_item, torch.Tensor):
            feature_map = fm_item
            kernel_radius = 1
            return feature_map, kernel_radius

        if isinstance(fm_item, (tuple, list)):
            if len(fm_item) >= 2 and isinstance(fm_item[1], torch.Tensor):
                feature_map = fm_item[1]
            else:
                raise TypeError(
                    "fm_list items must be torch.Tensors or tuples like (name, feature_map, kernel_size)."
                )

            kernel_radius = 1
            if len(fm_item) >= 3:
                k = fm_item[2]
                if isinstance(k, int):
                    kernel_radius = max(0, (k - 1) // 2)
                elif isinstance(k, (tuple, list)) and len(k) >= 1 and isinstance(k[0], int):
                    kernel_radius = max(0, (int(k[0]) - 1) // 2)

            return feature_map, kernel_radius

        raise TypeError(
            f"Unsupported fm_list item type: {type(fm_item).__name__}. Expected Tensor or tuple/list."
        )

    @staticmethod
    def _reduce_corr_matrix(corr_mat: torch.Tensor, correlation_loss: str) -> torch.Tensor:
        match correlation_loss:
            case "full":
                return corr_mat.sum()
            case "wta":
                corr_full = corr_mat + corr_mat.transpose(0, 1)
                ch_sums = corr_full.sum(dim=1)
                return ch_sums.max()
            case _:
                raise ValueError(
                    f"Unknown correlation_loss={correlation_loss!r}. Expected one of: 'full', 'wta'."
                )

    @staticmethod
    def luca_fn(
        feature_map: torch.Tensor,
        kernel_radius: int,
        mode: str = "max",
        eps: float = 1e-6,
    ) -> torch.Tensor:
        if mode not in {"mean", "max"}:
            raise ValueError(f"mode must be 'mean' or 'max', got {mode!r}")

        if feature_map.ndim != 4:
            raise ValueError(f"feature_map must be [B, C, H, W], got shape={tuple(feature_map.shape)}")

        B, C, H, W = feature_map.shape
        R = int(kernel_radius)
        if R < 0:
            raise ValueError(f"kernel_radius must be >= 0, got {R}")

        # 1) Z-score each coordinate across batch
        mu = feature_map.mean(dim=0, keepdim=True)
        sig = feature_map.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
        Z = (feature_map - mu) / sig  # [B, C, H, W]

        # 2) Extract neighborhood within `kernel_radius`
        patch_HW = (2 * R) + 1
        patch_len = patch_HW * patch_HW
        patches = F.unfold(Z, kernel_size=patch_HW, padding=R)  # [B, C * patch_len, H * W]
        patches = patches.view(B, C, patch_len, H * W)  # [B, C, patch_len, H * W]

        # 3) Center element in each neighborhood is (dh=0, dw=0)
        center_k = patch_len // 2
        center = patches[:, :, center_k, :]  # [B, C, H * W]

        # 4) Channel correlation matrix
        corr = torch.einsum("bip,bjkp->ijkp", center, patches) / B  # [C, C, patch_len, H * W]
        corr = corr * corr

        same_ch = torch.eye(C, dtype=torch.bool, device=corr.device)[:, :, None, None]
        center_pos = (torch.arange(patch_len, device=corr.device) == center_k)[None, None, :, None]
        corr = corr.masked_fill(same_ch & center_pos, 0)

        if mode == "max":
            corr = corr.amax(dim=(2, 3))
        elif mode == "mean":
            corr = corr.mean(dim=(2, 3))

        # 5) Keep only upper-triangle (exclude diagonal)
        mask = torch.triu(torch.ones((C, C), dtype=torch.bool, device=corr.device), diagonal=1)
        corr = corr * mask

        return corr

### Training

In [ ]:
def _load_config(config_or_path: dict | str | None) -> dict:
    if config_or_path is None:
        return {}
    if isinstance(config_or_path, dict):
        return dict(config_or_path)
    if isinstance(config_or_path, str):
        if not config_or_path.endswith(".json"):
            raise ValueError("Only .json config files are supported for now.")
        with open(config_or_path, "r", encoding="utf-8") as f:
            return json.load(f)
    raise TypeError(f"config_or_path must be dict | str | None, got {type(config_or_path).__name__}")

In [ ]:
def train_one_run(config_or_path: dict | str | None = None) -> dict:
    base_cfg = {
        "project": "cnn-redundancy-reduction",
        "architecture_type": "cnn",
        "learning_rate": 1e-3,
        "epochs": 10,
        "batch_size": 64,
        "lambda_strength": 0.0,
        "correlation_mode": "max",
        "correlation_loss": "full",
        "log_every_steps": 100,
    }
    base_cfg.update(_load_config(config_or_path))

    run = wandb.init(
        project=base_cfg["project"],
        config=base_cfg,
        name=(
            f"{base_cfg['architecture_type']}"
            f"-lam_{base_cfg['lambda_strength']}"
            f"-{base_cfg['correlation_mode']}"
            f"-{base_cfg['correlation_loss']}"
            f"-lr_{base_cfg['learning_rate']}"
        ),
    )

    cfg = run.config

    model_cls = get_autoencoder_type(cfg["architecture_type"])
    model = model_cls().to(DEVICE)

    criterion = RedundancyLoss(
        lambda_strength=cfg["lambda_strength"],
        correlation_mode=cfg["correlation_mode"],
        correlation_loss=cfg["correlation_loss"],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["learning_rate"])

    def _accuracy(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        return (logits.argmax(dim=1) == y).float().mean()

    def _eval() -> tuple[float, float]:
        model.eval()
        loss_sum = 0.0
        acc_sum = 0.0
        n = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                logits, convs = model(xb)
                loss = criterion(y=yb, y_pred=logits, fm_list=convs)
                bs = int(xb.shape[0])
                loss_sum += float(loss.detach().cpu()) * bs
                acc_sum += float(_accuracy(logits, yb).detach().cpu()) * bs
                n += bs
        return loss_sum / max(1, n), acc_sum / max(1, n)

    global_step = 0

    for epoch in range(int(cfg["epochs"])):
        model.train()

        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits, convs = model(xb)
            loss = criterion(y=yb, y_pred=logits, fm_list=convs)
            loss.backward()
            optimizer.step()

            if global_step % int(cfg["log_every_steps"]) == 0:
                wandb.log(
                    {
                        "train/loss": float(loss.detach().cpu()),
                        "train/ce": float(criterion.last_ce_loss.detach().cpu()),
                        "train/corr_total": float(criterion.last_corr_total.detach().cpu()),
                        "train/acc": float(_accuracy(logits, yb).detach().cpu()),
                        "epoch": epoch,
                        "step": global_step,
                    },
                    step=global_step,
                )

            global_step += 1

        test_loss, test_acc = _eval()
        wandb.log(
            {
                "test/loss": test_loss,
                "test/acc": test_acc,
                "epoch": epoch,
                "step": global_step,
            },
            step=global_step,
        )

    metrics = {"test/loss": test_loss, "test/acc": test_acc}
    run.finish()
    return metrics


In [ ]:
def _sweep_train() -> None:
    train_one_run(dict(wandb.config))


SWEEP_CONFIG = {
    "method": "grid",
    "metric": {"name": "test/acc", "goal": "maximize"},
    "parameters": {
        "lambda_strength": {"values": [0.0, 1e-3, 1e-2]},
        "architecture_type": {"values": ["cnn", "resnet"]},
        "learning_rate": {"value": 1e-3},
        "epochs": {"value": 10},
        "batch_size": {"value": 64},
        "correlation_mode": {"value": "max"},
        "correlation_loss": {"value": "full"},
        "log_every_steps": {"value": 100},
        "project": {"value": "cnn-redundancy-reduction"},
    },
}

sweep_id = wandb.sweep(SWEEP_CONFIG, project=SWEEP_CONFIG["parameters"]["project"]["value"])
wandb.agent(sweep_id, function=_sweep_train)